In [1]:
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install wikipedia
!pip install duckduckgo-search

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
  Using cached duckduckgo_search-8.1.1-py3-none-any.whl.metadata (16 kB)
Using cached duckduckgo_search-8.1.1-py3-none-any.whl (18 kB)
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.1 MB 3.5 MB/s eta 0:00:01
   --------------- ------------------------ 1.6/4.1 MB 3.6 MB/s eta 0:00:01
   ----------------------- ---------------- 2.4/4.1 MB 3.5 MB/s eta 0:00:01
   ------------------------------ ------

In [9]:
!pip install -U ddgs

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/6 [socksio]
   ------------- -------------------------- 2/6 [hpack]
   ------------- -------------------------- 2/6 [hpack]
   ------------- -------------------------- 2/6 [hpack]
   ------------- -------------------------- 2/6 [hpack]
   -------------------- ------------------- 3/6 [fake-useragent]
   -------------------- ------------------- 3/6 [fake-useragent]
   -------------------------- ------------- 4/6 [h2]
   -------------------------- ------------- 4/6 [h2]
   -------------------------- ------------- 4/6 [h2]
   -------------------------- ------------- 4/6 [h2]
   --------------------------------- ------ 5/6 [ddgs]
   --------------------------------- ------ 5/6 [ddgs]
   --------------------------------- ------ 5/6 [ddgs]
   --------------------------------- ------ 5/6 [ddgs]
   --------------------------------- ------ 5/6 [ddgs]
   -------------------

In [5]:
# =====================================================================
# STEP 1: DEPENDENCIES & SETUP
# =====================================================================
# !pip install langchain langchain-openai langchain-community duckduckgo-search pydantic

import os
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun
from datetime import date
from dotenv import load_dotenv

load_dotenv(override=True)

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# =====================================================================
# STEP 2: DEFINE THE TOOLS
# =====================================================================
@tool
def discover_products_tool(category_query: str) -> str:
    """Searches the web to find lists, roundups, or recommendations for products within a specific category or budget."""
    search = DuckDuckGoSearchRun()
    today_year = date.today()
    country = "India"
    try:
        return search.run(f"best {category_query} current models overview articles as of {today_year} in {country}")
    except Exception as e:
        return f"Error discovering products: {str(e)}"

@tool
def research_technical_specs(product_name: str) -> str:
    """Searches the web for official manufacturer documentation and hardware specifications for a specific product model."""
    search = DuckDuckGoSearchRun()
    try:
        return search.run(f"{product_name} official technical specifications datasheet hardware specs")
    except Exception as e:
        return f"Error fetching technical specs: {str(e)}"

@tool
def search_reviews(product_name: str) -> str:
    """Searches the live web to find user reviews, pros, and cons for a specific product model."""
    search = DuckDuckGoSearchRun()
    try:
        return search.run(f"{product_name} user reviews pros cons site:reddit.com OR site:techradar.com")
    except Exception as e:
        return f"Error fetching reviews: {str(e)}"

@tool
def check_market_price(product_name: str) -> str:
    """Searches the live web to find the current retail price or MRP of a specific product model."""
    search = DuckDuckGoSearchRun()
    try:
        return search.run(f"{product_name} current retail price MRP buy online")
    except Exception as e:
        return f"Error fetching price: {str(e)}"

# =====================================================================
# STEP 3: STRUCTURED SCHEMAS & AGENTS DEFINITION
# =====================================================================

# --- ROUTER COMPONENT ---
class PriceFilterIntent(BaseModel):
    has_budget_constraint: bool = Field(description="True if the user specifies a maximum price or budget limit.")
    max_budget_inr: Optional[float] = Field(description="The maximum INR amount specified by the user, if any.")

router_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze the user query and determine if they are looking for a product under a specific price or budget restriction."),
    ("human", "{user_query}")
])
router_chain = router_prompt | llm.with_structured_output(PriceFilterIntent)


# --- AGENT 0: PRODUCT DISCOVERY AGENT (NEW) ---
class DiscoveredProducts(BaseModel):
    category_confirmed: str = Field(description="The generic category of the products being searched.")
    product_names: List[str] = Field(description="A list of the top 2-3 specific, exact product model names found matching the user query.")

discovery_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a product discovery agent. Analyze the broad user request and search data to identify 2 to 3 exact product models that fit the criteria. Output only their specific names."),
    ("human", "User criteria: {user_query}. Search findings: {search_context}")
])
structured_discoverer = discovery_prompt | llm.with_structured_output(DiscoveredProducts)


# --- AGENT 1: TECHNICAL SPECS RESEARCHER ---
class TechnicalSpecs(BaseModel):
    product_name: str = Field(description="The exact official name of the product.")
    key_specs: List[str] = Field(description="List of core technical specifications found.")
    summary: str = Field(description="A brief 2-sentence summary of what this product is.")

researcher_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert technical researcher. Use the tool context to extract objective hardware/software facts for the given product model. Filter out marketing fluff."),
    ("human", "Extract specs for product model: {product_to_research}. Context data: {tool_context}")
])
structured_researcher = researcher_prompt | llm.with_structured_output(TechnicalSpecs)


# --- AGENT 1.5: PRICE GUARDRAIL AGENT ---
class BudgetEvaluation(BaseModel):
    estimated_market_price: float = Field(description="The numeric dollar value found for the product.")
    is_within_budget: bool = Field(description="True if estimated_market_price is less than or equal to the allowed budget.")
    price_notes: str = Field(description="A quick breakdown of pricing variants found.")

price_agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a pricing specialist. Look at the raw web search data, extract the numeric price of the product, and evaluate if it fits within the user's budget limit of INR {allowed_budget}."),
    ("human", "Evaluate pricing for product. Web pricing data: {price_tool_context}")
])
price_evaluator_chain = price_agent_prompt | llm.with_structured_output(BudgetEvaluation)


# --- AGENT 2: REVIEW ANALYST ---
class PublicSentiment(BaseModel):
    product_name: str
    pros: List[str] = Field(description="Top 3 pros mentioned by users.")
    cons: List[str] = Field(description="Top 3 cons or complaints mentioned by users.")
    overall_sentiment: str = Field(description="Is the reception Positive, Mixed, or Negative?")

review_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a consumer review analyst. Use the context to gather recent user feedback. Be objective and capture what real users are saying."),
    ("human", "Analyze reviews for: {product_name}. Web Data: {tool_context}")
])
structured_reviewer = review_prompt | llm.with_structured_output(PublicSentiment)


# --- AGENT 3: RECOMMENDATION ENGINE ---
class FinalVerdict(BaseModel):
    verdict: str = Field(description="Should the user buy this? (e.g., 'Highly Recommended', 'Over Budget - Avoid', 'Good Value Choice')")
    justification: str = Field(description="A clear explanation balancing specs, reviews, and budget constraints.")
    target_audience: str = Field(description="Who is this product best suited for?")

advisor_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an elite product advisor. Provide a definitive buying recommendation. If budget data is present, prioritize checking if the item is financially viable for the user."),
    ("human", "Technical Specs:\n{specs}\n\nUser Reviews:\n{reviews}\n\nOptional Budget/Price Check Data:\n{budget_data}")
])
structured_advisor = advisor_prompt | llm.with_structured_output(FinalVerdict)

# =====================================================================
# STEP 4: PIPELINE ORCHESTRATION FUNCTION
# =====================================================================
def run_discovery_and_research_pipeline(user_query: str):
    print(f"🔍 Analyzing Request: '{user_query}'\n")
    
    # --- Step 0A: Check Routing Intent for Budget ---
    routing_decision = router_chain.invoke({"user_query": user_query})
    
    # --- Step 0B: Discover Product Options ---
    print("🔎 Agent 0: Searching for specific product models matching your description...")
    discovery_raw_data = discover_products_tool.invoke({"category_query": user_query})
    discovered_data = structured_discoverer.invoke({
        "user_query": user_query,
        "search_context": discovery_raw_data
    })
    
    print(f"🎯 Discovered candidates to analyze: {discovered_data.product_names}\n")
    
    # --- Loop Over Discovered Products Sequentially ---
    for product in discovered_data.product_names:
        print(f"--- Processing: {product} ---")
        
        # --- Step 1: Specs Researcher ---
        print(f"  🕵️‍♂️ Agent 1: Fetching technical specifications...")
        spec_raw_data = research_technical_specs.invoke({"product_name": product})
        specs_output = structured_researcher.invoke({
            "product_to_research": product, 
            "tool_context": spec_raw_data
        })

        # --- Step 1.5: Conditional Budget Agent ---
        budget_report_json = "{}" 
        if routing_decision.has_budget_constraint:
            print(f"  💰 Agent 1.5: Running Price Check against budget: INR {routing_decision.max_budget_inr}...")
            price_raw_data = check_market_price.invoke({"product_name": specs_output.product_name})
            budget_analysis = price_evaluator_chain.invoke({
                "allowed_budget": routing_decision.max_budget_inr,
                "price_tool_context": price_raw_data
            })
            print(f"     ↳ Est. Price: INR {budget_analysis.estimated_market_price} | Within Budget: {budget_analysis.is_within_budget}")
            budget_report_json = budget_analysis.model_dump_json()

        # --- Step 2: Review Analyst ---
        print("  📊 Agent 2: Gathering user sentiment and reviews...")
        review_tool_result = search_reviews.invoke({"product_name": specs_output.product_name})
        reviews_output = structured_reviewer.invoke({
            "product_name": specs_output.product_name, 
            "tool_context": review_tool_result
        })

        # --- Step 3: Advisor Verdict ---
        print("  ⚖️ Agent 3: Generating final verdict...")
        final_recommendation = structured_advisor.invoke({
            "specs": specs_output.model_dump_json(),
            "reviews": reviews_output.model_dump_json(),
            "budget_data": budget_report_json
        })

        # --- Print Final Summary Output For This Product ---
        print(f"  📋 VERDICT FOR {specs_output.product_name.upper()}:")
        print(f"     Status:        {final_recommendation.verdict}")
        print(f"     Target User:   {final_recommendation.target_audience}")
        print(f"     Reasoning:     {final_recommendation.justification}")
        print("-" * 60 + "\n")

# =====================================================================
# STEP 5: TEST THE PIPELINE
# =====================================================================
run_discovery_and_research_pipeline("Best phones between 90k to 1L INR")

🔍 Analyzing Request: 'Best phones between 90k to 1L INR'

🔎 Agent 0: Searching for specific product models matching your description...
🎯 Discovered candidates to analyze: ['Samsung Galaxy S23', 'OnePlus 11', 'Google Pixel 7 Pro']

--- Processing: Samsung Galaxy S23 ---
  🕵️‍♂️ Agent 1: Fetching technical specifications...
  💰 Agent 1.5: Running Price Check against budget: INR 100000.0...
     ↳ Est. Price: INR 95999.0 | Within Budget: True
  📊 Agent 2: Gathering user sentiment and reviews...
  ⚖️ Agent 3: Generating final verdict...
  📋 VERDICT FOR SAMSUNG GALAXY S23:
     Status:        Highly Recommended
     Target User:   This product is best suited for tech enthusiasts, photography lovers, and users looking for a compact yet powerful smartphone.
     Reasoning:     The Samsung Galaxy S23 offers top-tier performance with its Snapdragon 8 Gen 2 processor and a versatile camera system, making it an excellent choice for users who prioritize both performance and photography. The posit

In [9]:
check_market_price

StructuredTool(name='check_market_price', description='Searches the live web to find the current retail price or MRP of a specific product model.', args_schema=<class 'langchain_core.utils.pydantic.check_market_price'>, func=<function check_market_price at 0x0000021647214CA0>)

In [36]:
routing_decision

NameError: name 'routing_decision' is not defined

In [40]:
user_query  = "Gaming laptops under INR 135000"
routing_decision = router_chain.invoke({"user_query": user_query})

In [43]:
routing_decision.model_dump_json()

'{"has_budget_constraint":true,"max_budget_inr":135000.0}'

In [44]:
discovery_raw_data = discover_products_tool.invoke({"category_query": user_query})

In [45]:
discovery_raw_data

'Get the best deals on Laptops & Netbooks from the largest online selection at eBay.Good - Refurbished · Dell. $277.31. $40.61 shipping. Shop a wide selection of Laptops including 2 in 1 and traditional laptops at Amazon.com. Free shipping and free returns on eligible items. Shop Laptops online from top brands like HP, Dell, Apple, Lenovo & Asus at Croma. Get the best price, No Cost EMI, and easy exchange with AI available features. In this article, we’ll look at the best monitors under Rs. 10,000. At this price segment, you get some decent options. Below, you’ll find the entries for some of the best monitors under INR 10000. Let’s take a look at them! Acer Ha220Q 21.5 Inch Monitor With Stereo Speakers. Which is the best gaming phone under 20000 in 2026? The Vivo T5x 5G offers the strongest raw performance and biggest battery, while CMF Phone 2 Pro provides better 120FPS BGMI support for competitive gaming.'

In [46]:
discovered_data = structured_discoverer.invoke({
        "user_query": user_query,
        "search_context": discovery_raw_data
    })

In [47]:
discovered_data

DiscoveredProducts(category_confirmed='Gaming Laptops', product_names=['ASUS TUF Gaming F15', 'Acer Nitro 5', 'HP Pavilion Gaming 15'])

In [49]:
product = discovered_data.product_names[0]
spec_raw_data = research_technical_specs.invoke({"product_name": product})
specs_output = structured_researcher.invoke({
    "product_to_research": product, 
    "tool_context": spec_raw_data
})

In [50]:
specs_output

TechnicalSpecs(product_name='ASUS TUF Gaming F15 FX507VU-LP163W', key_specs=['Intel Core i7-12700H processor', '16GB RAM', '512GB SSD', '15.6-inch FHD display', 'NVIDIA GeForce RTX 3060 graphics', 'Windows 11 operating system'], summary='The ASUS TUF Gaming F15 is a gaming laptop designed for performance with its powerful Intel Core i7 processor and dedicated NVIDIA graphics. It features a 15.6-inch FHD display, making it suitable for gaming and content creation.')

In [51]:
price_raw_data = check_market_price.invoke({"product_name": specs_output.product_name})
price_raw_data

'March 30, 2026 - ASUS TUF Gaming F15 (FX507VU-LP163W) F15 Jump right into the action with the TUF Gaming F15Multitask with ease thanks to the Intel Core i7-13620H processor.And with 16GB of ultra-fast RAM on Windows 11Take advantage of the full gaming performance of the GeForce RTX 4050 laptop GPU. March 28, 2026 - ASUS TUF Gaming F15 FX507VV with 13th Gen i7, RTX 4060 & fast SSD. Ideal for gaming & creators. Limited stock. Buy now. July 22, 2025 - لابتوب الألعاب ASUS TUF F15 FX507VU-LP163W يعتبر جهازًا متطورًا مخصصًا لعشاق الألعاب والمستخدمين الذين يتطلعون إلى أداء عالي الجودة في تشغيل التطبيقات الثقيلة والمتطلبات الرسومية العالية. 1 week ago - It would potentially help you understand how Asus TUF A16 FA607NUQ-RL047WS Gaming L… stands against Asus TUF Gaming F15 FX507VV-LP487WS Ga… and which one should you buy The current lowest price found for Asus TUF A16 FA607NUQ-RL047WS Gaming L… is ₹99,990 and for Asus TUF Gaming F15 ... December 6, 2025 - ASUS TUF Gaming F15 Laptop Delhi Nehru 

In [52]:
specs_output = structured_researcher.invoke({
            "product_to_research": product, 
            "tool_context": spec_raw_data
        })

In [53]:
specs_output

TechnicalSpecs(product_name='ASUS TUF Gaming F15 FX507VU-LP163W', key_specs=['Intel Core i7-12700H processor', '16GB RAM', '512GB SSD', 'Windows 11 operating system', '15.6-inch FHD display', 'NVIDIA GeForce RTX 3060 graphics card'], summary='The ASUS TUF Gaming F15 is a gaming laptop designed for performance with its powerful Intel Core i7 processor and dedicated NVIDIA graphics. It features a 15.6-inch FHD display and ample RAM and storage for gaming and content creation.')

In [54]:
budget_analysis = price_evaluator_chain.invoke({
                "allowed_budget": routing_decision.max_budget_inr,
                "price_tool_context": price_raw_data
            })

In [55]:
budget_analysis

BudgetEvaluation(estimated_market_price=99990.0, is_within_budget=True, price_notes='The lowest price found for the ASUS TUF Gaming F15 is ₹99,990, which is below the budget limit of ₹135,000.')

In [64]:
check_market_price

StructuredTool(name='check_market_price', description='Searches the live web to find the current retail price or MRP of a specific product model.', args_schema=<class 'langchain_core.utils.pydantic.check_market_price'>, func=<function check_market_price at 0x0000016BFF1A8720>)

In [2]:
class Person1(BaseModel):
    name: str
    age: int
    city: str

In [11]:
person  = Person1(name = "Srimanth", age = 30, city = "bangalore")

In [4]:
person

Person1(name='Srimanth', age=30, city='bangalore')